# Movie Recommender System
> A recommender system, also known as a recommendation system, is a subclass of information filtering systems that seeks to predict the “rating” or “preference” a user would give to an item. 

![](https://cdn.prod.website-files.com/670cbf146221ee06c3cdd761/67120d05aeb7b880cef08357_movie%20recommendation%20system.webp)

## Types of Recommendation Systems
### 1. Content-Based Filtering
Recommends items similar to those the user has liked before, based on item features (e.g., genre, keywords, tags).

### 2. Collaborative Filtering
Recommends items based on the preferences of similar users. It can be:

* User-based: Find users with similar preferences.

* Item-based: Find items similar to those the user has liked.

### 3. Hybrid Systems
Combine content-based and collaborative filtering to leverage the strengths of both methods and improve recommendations.

![](https://ik.imagekit.io/upgrad1/abroad-images/imageCompo/images/1718046849736_Untitled_design_2024_06_11T004357AT1WS4.webp?pr-true)

## Importing Libraries

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [2]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

## Jacard Similarity function
> Jaccard similarity refers to a measure of similarity between two sets of keywords. It is used to determine if clusters containing raw messages are duplicates by computing the similarity score between the sets of keywords associated with each cluster.

![Jacard](https://i.ytimg.com/vi/Ah_4xqvS1WU/maxresdefault.jpg)

In [3]:
def jaccard_similarity(list1, list2):
    set1 = set(list1)
    set2 = set(list2)
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return float(intersection/union) if union !=0 else 0

## Reading the csv file

In [4]:
df_main = pd.read_csv("/kaggle/input/tmdb-movies-daily-updates/TMDB_all_movies.csv")
df_main.head()

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,budget,imdb_id,original_language,original_title,overview,popularity,tagline,genres,production_companies,production_countries,spoken_languages,cast,director,director_of_photography,writers,producers,music_composer,imdb_rating,imdb_votes,poster_path
0,2,Ariel,7.111,346.0,Released,1988-10-21,0.0,73.0,0.0,tt0094675,fi,Ariel,A Finnish man goes to the city to find a job a...,1.4744,NaN,"Comedy, Drama, Romance, Crime",Villealfa Filmproductions,Finland,suomi,"Turo Pajala, Matti Jaaranen, Marja Packalén, J...",Aki Kaurismäki,Timo Salminen,Aki Kaurismäki,Aki Kaurismäki,NaN,7.4,9122.0,/ojDg0PGvs6R9xYFodRct2kdI6wC.jpg
1,3,Shadows in Paradise,7.293,409.0,Released,1986-10-17,0.0,74.0,0.0,tt0092149,fi,Varjoja paratiisissa,"Nikander, a rubbish collector and would-be ent...",1.9295,NaN,"Comedy, Drama, Romance",Villealfa Filmproductions,Finland,"suomi, English, svenska","Esko Nikkari, Mari Rantasila, Marina Martinoff...",Aki Kaurismäki,Timo Salminen,Aki Kaurismäki,Mika Kaurismäki,NaN,7.4,7937.0,/nj01hspawPof0mJmlgfjuLyJuRN.jpg
2,5,Four Rooms,5.900,2685.0,Released,1995-12-09,4257354.0,98.0,4000000.0,tt0113101,en,Four Rooms,It's Ted the Bellhop's first night on the job....,2.3746,Twelve outrageous guests. Four scandalous requ...,"Comedy, Crime","Miramax, A Band Apart",United States of America,English,"Tamlyn Tomita, Tim Roth, Madonna, Paul Caldero...","Robert Rodriguez, Quentin Tarantino, Alexandre...","Andrzej Sekula, Rodrigo García, Phil Parmet, G...","Robert Rodriguez, Quentin Tarantino, Alexandre...","Lawrence Bender, Quentin Tarantino, Alexandre ...",Combustible Edison,6.7,113944.0,/75aHn1NOYXh4M7L5shoeQ6NGykP.jpg
3,6,Judgment Night,6.458,349.0,Released,1993-10-15,12136938.0,109.0,21000000.0,tt0107286,en,Judgment Night,"Four young friends, while taking a shortcut en...",1.3384,Don't move. Don't whisper. Don't even breathe.,"Action, Crime, Thriller","Largo Entertainment, JVC, Universal Pictures",United States of America,English,"Stephen Dorff, Deirdre Kelly, Emilio Estevez, ...",Stephen Hopkins,Peter Levy,"Lewis Colick, Jere Cunningham","Marilyn Vance, Gene Levy, Lloyd Segan",Alan Silvestri,6.6,19849.0,/3rvvpS9YPM5HB2f4HYiNiJVtdam.jpg
4,8,Life in Loops (A Megacities RMX),7.500,27.0,Released,2006-01-01,0.0,80.0,42000.0,tt0825671,en,Life in Loops (A Megacities RMX),Timo Novotny labels his new project an experim...,3.2030,A Megacities remix.,Documentary,inLoops,Austria,"English, हिन्दी, 日本語, Pусский, Español",NaN,Timo Novotny,Wolfgang Thaler,"Michael Glawogger, Timo Novotny","Ulrich Gehmacher, Timo Novotny",NaN,8.2,284.0,/7ln81BRnPR2wqxuITZxEciCe1lc.jpg


## Selecting the desired features

In [5]:
df = df_main[['id', 'title', 'overview', 'genres', 'production_companies', 'cast', 'director', 'writers', 'producers']]

Next we will deal with Null values and also convert some of the columns to list

In [6]:
cols_to_lower = ['director', 'writers', 'producers', 'cast', 'production_companies', 'genres']
for col in cols_to_lower:
    df[col] = df[col].fillna('').str.lower().str.replace(r"\s+", "", regex=True).str.split(",")
    df[col] = df[col].apply(lambda x: x[:4] if len(x)>4 else x)


df["overview"] = df["overview"].fillna('').str.lower()

/tmp/ipykernel_13/2146218560.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].fillna('').str.lower().str.replace(r"\s+", "", regex=True).str.split(",")
/tmp/ipykernel_13/2146218560.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].apply(lambda x: x[:4] if len(x)>4 else x)
/tmp/ipykernel_13/2146218560.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveat

In [7]:
df.head()

,id,title,overview,genres,production_companies,cast,director,writers,producers
0,2,Ariel,a finnish man goes to the city to find a job a...,"[comedy, drama, romance, crime]",[villealfafilmproductions],"[turopajala, mattijaaranen, marjapackalén, jyr...",[akikaurismäki],[akikaurismäki],[akikaurismäki]
1,3,Shadows in Paradise,"nikander, a rubbish collector and would-be ent...","[comedy, drama, romance]",[villealfafilmproductions],"[eskonikkari, marirantasila, marinamartinoff, ...",[akikaurismäki],[akikaurismäki],[mikakaurismäki]
2,5,Four Rooms,it's ted the bellhop's first night on the job....,"[comedy, crime]","[miramax, abandapart]","[tamlyntomita, timroth, madonna, paulcalderon]","[robertrodriguez, quentintarantino, alexandrer...","[robertrodriguez, quentintarantino, alexandrer...","[lawrencebender, quentintarantino, alexandrero..."
3,6,Judgment Night,"four young friends, while taking a shortcut en...","[action, crime, thriller]","[largoentertainment, jvc, universalpictures]","[stephendorff, deirdrekelly, emilioestevez, ro...",[stephenhopkins],"[lewiscolick, jerecunningham]","[marilynvance, genelevy, lloydsegan]"
4,8,Life in Loops (A Megacities RMX),timo novotny labels his new project an experim...,[documentary],[inloops],[],[timonovotny],"[michaelglawogger, timonovotny]","[ulrichgehmacher, timonovotny]"


In [8]:
df.cast[0]

['turopajala', 'mattijaaranen', 'marjapackalén', 'jyrkiolsonen']

In [9]:
cols_to_merge = ['director', 'writers', 'producers', 'cast', 'production_companies', 'genres']

# Function to join lists from multiple columns, convert to set (for uniqueness), then back to list
def merge_and_unique(row):
    combined = []
    for col in cols_to_merge:
        combined.extend(row[col])  # Add the list from each column to combined list
    return list(set(combined))  # Convert to set for uniqueness, then back to list

# Apply the function to each row
df['tags'] = df.apply(merge_and_unique, axis=1)

/tmp/ipykernel_13/2250808477.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tags'] = df.apply(merge_and_unique, axis=1)


In [10]:
df.drop(columns=cols_to_merge, inplace=True)

/tmp/ipykernel_13/708697862.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(columns=cols_to_merge, inplace=True)


In [11]:
df.set_index('id', inplace=True)

In [12]:
df.head()

,title,overview,tags
id,,,
2,Ariel,a finnish man goes to the city to find a job a...,"[mattijaaranen, comedy, turopajala, crime, dra..."
3,Shadows in Paradise,"nikander, a rubbish collector and would-be ent...","[marirantasila, comedy, drama, akikaurismäki, ..."
5,Four Rooms,it's ted the bellhop's first night on the job....,"[comedy, crime, allisonanders, quentintarantin..."
6,Judgment Night,"four young friends, while taking a shortcut en...","[stephendorff, lloydsegan, jvc, universalpictu..."
8,Life in Loops (A Megacities RMX),timo novotny labels his new project an experim...,"[, timonovotny, michaelglawogger, documentary,..."


In [13]:
df['tags'] = df['tags'].apply(lambda x: [item for item in x if item != ''] if isinstance(x, list) else x)
df.head()

/tmp/ipykernel_13/1518854655.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tags'] = df['tags'].apply(lambda x: [item for item in x if item != ''] if isinstance(x, list) else x)


,title,overview,tags
id,,,
2,Ariel,a finnish man goes to the city to find a job a...,"[mattijaaranen, comedy, turopajala, crime, dra..."
3,Shadows in Paradise,"nikander, a rubbish collector and would-be ent...","[marirantasila, comedy, drama, akikaurismäki, ..."
5,Four Rooms,it's ted the bellhop's first night on the job....,"[comedy, crime, allisonanders, quentintarantin..."
6,Judgment Night,"four young friends, while taking a shortcut en...","[stephendorff, lloydsegan, jvc, universalpictu..."
8,Life in Loops (A Megacities RMX),timo novotny labels his new project an experim...,"[timonovotny, michaelglawogger, documentary, i..."


In [14]:
df["tags"][8]

['timonovotny',
 'michaelglawogger',
 'documentary',
 'inloops',
 'ulrichgehmacher']

## Droping the Duplicates

In [15]:
# Count the number of duplicates
df['title'].duplicated().sum()

165494

In [16]:
df = df.drop_duplicates(subset=['title'], keep='first')

In [17]:
df[df['title'].duplicated()]

,title,overview,tags
id,,,


In [18]:
df.shape

(917020, 3)

In [19]:
df.head()

,title,overview,tags
id,,,
2,Ariel,a finnish man goes to the city to find a job a...,"[mattijaaranen, comedy, turopajala, crime, dra..."
3,Shadows in Paradise,"nikander, a rubbish collector and would-be ent...","[marirantasila, comedy, drama, akikaurismäki, ..."
5,Four Rooms,it's ted the bellhop's first night on the job....,"[comedy, crime, allisonanders, quentintarantin..."
6,Judgment Night,"four young friends, while taking a shortcut en...","[stephendorff, lloydsegan, jvc, universalpictu..."
8,Life in Loops (A Megacities RMX),timo novotny labels his new project an experim...,"[timonovotny, michaelglawogger, documentary, i..."


In [20]:
df.isnull().sum()

title       1
overview    0
tags        0
dtype: int64

In [21]:
df.dropna(inplace=True)

As the dataset is too big we will be using a portion of it

In [22]:
df=df[:30000]

## Tfidf Vectorizer
![](https://www.kdnuggets.com/wp-content/uploads/awan_convert_text_documents_tfidf_matrix_tfidfvectorizer_3.png)

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import nltk

# Ensure necessary NLTK data is downloaded
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')

# Initialize lemmatizer and stop words
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Define custom tokenizer
def lemmatize_and_remove_stopwords(text):
    tokens = word_tokenize(text.lower())
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in tokens]
    filtered_tokens = [token for token in lemmatized_tokens if token not in stop_words]
    return filtered_tokens

# Initialize TfidfVectorizer with custom tokenizer
vectorizer = TfidfVectorizer(tokenizer=lemmatize_and_remove_stopwords, max_features=5000)

tfidf_matrix = vectorizer.fit_transform(df["overview"])

# To view the feature names
print(len(vectorizer.get_feature_names_out()))



[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:528: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


5000


## Consine Similarity




![](https://builtin.com/sites/www.builtin.com/files/styles/ckeditor_optimize/public/inline-images/2_cosine-similarity.png)

I used the linear kernel to calculate the cosine similarity because the vectors formed by the tfidfvectorizer are normalized so using linear kernel or cosine similarity wouldnot make any difference at all.
But using Linear Kernel improved the speed of the Process

In [24]:
cosineSimilarities = linear_kernel(tfidf_matrix, tfidf_matrix)

In [25]:
print(cosineSimilarities.shape)


(30000, 30000)


In [26]:
print(vectorizer.get_feature_names_out())


['!' '#' '$' ... '“' '”' '•']


In [27]:
def combined_similarity(idx, cos_sim_matrix, df):
    # Get the actual position of the given index
    idx_pos = df.index.get_loc(idx)

    results = []
    for i in df.index:
        i_pos = df.index.get_loc(i)

        # Calculate similarities
        jac = jaccard_similarity(df['tags'][idx], df['tags'][i])
        cos = cos_sim_matrix[idx_pos][i_pos]

        combined = (cos * 0.5 + jac * 0.5)
        results.append((i, combined))  # i is the actual index label
    
    return sorted(results, key=lambda x: x[1], reverse=True)[:11]


In [28]:
recommendations = combined_similarity(idx=122, cos_sim_matrix=cosineSimilarities, df=df)

In [29]:
recommendations

[(122, 1.0),
 (121, 0.42118589300008136),
 (120, 0.3370049533759517),
 (1361, 0.17855869591632847),
 (123, 0.17783758639927888),
 (17832, 0.16532044905494547),
 (14763, 0.1413387751710264),
 (3040, 0.1373097549652993),
 (13194, 0.13185967760971407),
 (28058, 0.13169556871890487),
 (45678, 0.12971369273050953)]

In [30]:
for pair in recommendations:
    print(df.loc[pair[0]].title)

The Lord of the Rings: The Return of the King
The Lord of the Rings: The Two Towers
The Lord of the Rings: The Fellowship of the Ring
The Return of the King
The Lord of the Rings
Kull the Conqueror
Shadowless Sword
Night Watch
Highlander: The Search for Vengeance
Hercules in the Haunted World
Hot News


## Saving the cosine matrix and dataframe

In [31]:
import pickle

with open('cosine_sim_matrix.pkl', 'wb') as f:
    pickle.dump(cosineSimilarities, f)

In [32]:
df.to_csv('movies.csv')